# **Laboratorio 2 - Complejidad y búsqueda de hiperparámetros**

Laura Sanchez Bernal - 202411353

Baruc Jeronimo Triana Bastidas - 202416093


## 1. Importación de las librerías

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, validation_curve, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.utils import resample
import matplotlib.pyplot as plt


## 2. Carga de los datos

Se trabaja con la versión del conjunto de datos ya limpia y preparada en el Laboratorio 1 (`datatransf`), exportada como CSV. Por tanto, en este laboratorio el énfasis está en el modelado, la validación y el análisis del desempeño predictivo, no en la exploración ni el procesamiento de los datos.

In [ ]:
datatransf = pd.read_csv('./data/datos_limpios_lab1.csv')
data = datatransf.copy()
data.head()


## 3. Partición de los datos

Se mantiene el mismo `test_size` y `random_state` utilizados en el Laboratorio 1, de manera que los resultados de ambos laboratorios sean comparables.

In [ ]:
target = "temp_max_manana"
X = data.drop(columns=[target])
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## 4. Identificación de variables y preprocesamiento base

Se conservan las mismas variables categóricas identificadas en el Modelo 2 del Laboratorio 1 (el modelo seleccionado, que incluye las variables de ingeniería de características). Las variables numéricas se imputan con la mediana (por la presencia de outliers) y se escalan; las categóricas se imputan con la moda y se codifican mediante One-Hot Encoding.

In [ ]:
categorical_features = ['estacion_anio', 'mes', 'sector_viento']

numeric_features = [
    columna for columna in X_train.columns
    if columna not in categorical_features
]

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))


In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
])

# Preprocesamiento base (sin transformación polinomial), usado en Ridge y Lasso
numeric_transformer_base = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_base = ColumnTransformer(transformers=[
    ("num", numeric_transformer_base, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


## 5. Actividad 1 — Modelo de regresión polinomial

Se incorpora `PolynomialFeatures` dentro del pipeline numérico, después de la imputación y el escalamiento. La búsqueda de hiperparámetros se realiza con `GridSearchCV`, explorando el grado del polinomio y distintas estrategias de escalamiento.

In [ ]:
numeric_transformer_polinomial = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2)),
])

preprocessor_polinomial = ColumnTransformer(transformers=[
    ("num", numeric_transformer_polinomial, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

pipeline_regresion_polinomial = Pipeline(steps=[
    ("preprocesamiento", preprocessor_polinomial),
    ("modelo", LinearRegression()),
])


### Modelo de grado 2 (referencia inicial)

In [ ]:
pipeline_regresion_polinomial.fit(X_train, y_train)

modelo = pipeline_regresion_polinomial.named_steps["modelo"]
print("Número de coeficientes:", len(modelo.coef_))

y_train_pred = pipeline_regresion_polinomial.predict(X_train)
print('------ Regresión polinomial grado 2 - Entrenamiento ----')
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"R²: {r2_score(y_train, y_train_pred):.4f}")

y_test_pred = pipeline_regresion_polinomial.predict(X_test)
print('------ Regresión polinomial grado 2 - Test ----')
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")
print(f"R²: {r2_score(y_test, y_test_pred):.4f}")


### Modelo de grado 3 (para observar el efecto de la complejidad)

In [ ]:
numeric_transformer_g3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=3)),
])
preprocessor_g3 = ColumnTransformer(transformers=[
    ("num", numeric_transformer_g3, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
pipeline_g3 = Pipeline(steps=[
    ("preprocesamiento", preprocessor_g3),
    ("modelo", LinearRegression()),
])

pipeline_g3.fit(X_train, y_train)
print("Número de coeficientes:", len(pipeline_g3.named_steps["modelo"].coef_))

y_train_pred_g3 = pipeline_g3.predict(X_train)
y_test_pred_g3 = pipeline_g3.predict(X_test)
print('------ Grado 3 - Entrenamiento ----')
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred_g3)):.4f}, R²: {r2_score(y_train, y_train_pred_g3):.4f}")
print('------ Grado 3 - Test ----')
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred_g3)):.4f}, R²: {r2_score(y_test, y_test_pred_g3):.4f}")


### Búsqueda de hiperparámetros (grado + estrategia de escalamiento)

El enunciado pide explorar tanto el grado del polinomio como distintas estrategias de escalamiento dentro del pipeline.

In [ ]:
param_grid_poly = {
    "preprocesamiento__num__polynomial__degree": [1, 2, 3],
    "preprocesamiento__num__scaler": [StandardScaler(), RobustScaler(), MinMaxScaler()],
}

grid_poly = GridSearchCV(
    estimator=pipeline_regresion_polinomial,
    param_grid=param_grid_poly,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)


In [ ]:
%%time
grid_poly.fit(X_train, y_train)
print("Mejor configuración:", grid_poly.best_params_)


In [ ]:
mejor_modelo_polinomial = grid_poly.best_estimator_
y_test_pred = mejor_modelo_polinomial.predict(X_test)

print('------ Mejor modelo polinomial - Resultados en test ----')
rmse_poly_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_poly_test = mean_absolute_error(y_test, y_test_pred)
r2_poly_test = r2_score(y_test, y_test_pred)
print(f"RMSE: {rmse_poly_test:.4f}")
print(f"MAE: {mae_poly_test:.4f}")
print(f"R²: {r2_poly_test:.4f}")


## 6. Actividad 2 — Curvas de validación

Se utiliza `validation_curve` para observar cómo cambia el error del modelo (en entrenamiento y en validación cruzada) a medida que aumenta el grado de la transformación polinomial. Esto permite visualizar el trade-off sesgo-varianza: un grado bajo genera alto sesgo, mientras que un grado alto genera alta varianza (sobreajuste).

Nota: el rango de grados se limita a [1, 2, 3] — igual al explorado en la búsqueda de hiperparámetros — dado el número de variables numéricas del conjunto de datos; un grado mayor generaría un número de combinaciones polinomiales excesivamente alto.

In [ ]:
param_range = [1, 2, 3]

train_scores, cv_scores = validation_curve(
    estimator=pipeline_regresion_polinomial,
    X=X_train,
    y=y_train,
    param_name="preprocesamiento__num__polynomial__degree",
    param_range=param_range,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

train_rmse = -train_scores
cv_rmse = -cv_scores

train_rmse_mean = train_rmse.mean(axis=1)
train_rmse_std = train_rmse.std(axis=1)
cv_rmse_mean = cv_rmse.mean(axis=1)
cv_rmse_std = cv_rmse.std(axis=1)

print("RMSE promedio en entrenamiento:", train_rmse_mean)
print("RMSE promedio en validación cruzada:", cv_rmse_mean)
print("Desviación estándar en validación cruzada:", cv_rmse_std)


In [ ]:
plt.figure(figsize=(8,5))

plt.plot(param_range, train_rmse_mean, 'o-', color='tab:blue', label='Entrenamiento')
plt.fill_between(param_range, train_rmse_mean - train_rmse_std, train_rmse_mean + train_rmse_std,
                  alpha=0.2, color='tab:blue')

plt.plot(param_range, cv_rmse_mean, 'o-', color='tab:orange', label='Validación cruzada')
plt.fill_between(param_range, cv_rmse_mean - cv_rmse_std, cv_rmse_mean + cv_rmse_std,
                  alpha=0.2, color='tab:orange')

plt.xlabel('Grado del polinomio')
plt.ylabel('RMSE')
plt.title('Curva de validación - Regresión polinomial')
plt.xticks(param_range)
plt.legend()
plt.show()


_(Completa aquí la interpretación de la curva: en qué grado empieza a evidenciarse sobreajuste y por qué, en términos del trade-off sesgo-varianza.)_

## 7. Actividad 3 — Regresión regularizada (Ridge y Lasso)

Se definen un objeto `KFold` para la validación cruzada y los pipelines de Ridge y Lasso, cada uno con su propio espacio de búsqueda para el hiperparámetro de penalización `alpha`. Como referencia de comparación, también se entrena un modelo lineal sin regularización, con el mismo preprocesamiento base.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)


### Modelo sin regularización (referencia)

In [ ]:
pipeline_lineal = Pipeline(steps=[
    ("preprocesamiento", preprocessor_base),
    ("modelo", LinearRegression()),
])
pipeline_lineal.fit(X_train, y_train)

y_pred_lineal_train = pipeline_lineal.predict(X_train)
y_pred_lineal_test = pipeline_lineal.predict(X_test)

rmse_lineal_train = np.sqrt(mean_squared_error(y_train, y_pred_lineal_train))
mae_lineal_train = mean_absolute_error(y_train, y_pred_lineal_train)
r2_lineal_train = r2_score(y_train, y_pred_lineal_train)

rmse_lineal_test = np.sqrt(mean_squared_error(y_test, y_pred_lineal_test))
mae_lineal_test = mean_absolute_error(y_test, y_pred_lineal_test)
r2_lineal_test = r2_score(y_test, y_pred_lineal_test)

print(f"Train -> RMSE: {rmse_lineal_train:.4f}, MAE: {mae_lineal_train:.4f}, R²: {r2_lineal_train:.4f}")
print(f"Test  -> RMSE: {rmse_lineal_test:.4f}, MAE: {mae_lineal_test:.4f}, R²: {r2_lineal_test:.4f}")


### Ridge (L2)

In [ ]:
pipeline_ridge = Pipeline(steps=[
    ("preprocesamiento", preprocessor_base),
    ("modelo", Ridge()),
])

param_grid_ridge = {
    "modelo__alpha": [0.01, 0.1, 1, 10, 50, 100]
}

grid_ridge = GridSearchCV(
    estimator=pipeline_ridge, param_grid=param_grid_ridge,
    cv=kfold, scoring="neg_root_mean_squared_error", n_jobs=-1
)


In [ ]:
%%time
grid_ridge.fit(X_train, y_train)
print("Mejor alpha Ridge:", grid_ridge.best_params_)


In [ ]:
best_ridge = grid_ridge.best_estimator_

y_pred_ridge_train = best_ridge.predict(X_train)
y_pred_ridge_test = best_ridge.predict(X_test)

rmse_ridge_train = np.sqrt(mean_squared_error(y_train, y_pred_ridge_train))
mae_ridge_train = mean_absolute_error(y_train, y_pred_ridge_train)
r2_ridge_train = r2_score(y_train, y_pred_ridge_train)

rmse_ridge_test = np.sqrt(mean_squared_error(y_test, y_pred_ridge_test))
mae_ridge_test = mean_absolute_error(y_test, y_pred_ridge_test)
r2_ridge_test = r2_score(y_test, y_pred_ridge_test)

print(f"Train -> RMSE: {rmse_ridge_train:.4f}, MAE: {mae_ridge_train:.4f}, R²: {r2_ridge_train:.4f}")
print(f"Test  -> RMSE: {rmse_ridge_test:.4f}, MAE: {mae_ridge_test:.4f}, R²: {r2_ridge_test:.4f}")


#### Revisión de coeficientes — Ridge

In [ ]:
feature_names_ridge = best_ridge.named_steps["preprocesamiento"].get_feature_names_out()
coefs_ridge = best_ridge.named_steps["modelo"].coef_

coef_ridge_df = pd.DataFrame({
    "Variable": feature_names_ridge,
    "coef_ridge": coefs_ridge
})
coef_ridge_df["coef_ridge"] = coef_ridge_df["coef_ridge"].round(4)
coef_ridge_df = coef_ridge_df.sort_values("coef_ridge", key=abs, ascending=False)
coef_ridge_df.head(15)


### Lasso (L1)

In [ ]:
pipeline_lasso = Pipeline(steps=[
    ("preprocesamiento", preprocessor_base),
    ("modelo", Lasso(max_iter=5000)),
])

param_grid_lasso = {
    "modelo__alpha": [0.001, 0.01, 0.1, 0.5, 1, 5]
}

grid_lasso = GridSearchCV(
    estimator=pipeline_lasso, param_grid=param_grid_lasso,
    cv=kfold, scoring="neg_root_mean_squared_error", n_jobs=-1
)


In [ ]:
%%time
grid_lasso.fit(X_train, y_train)
print("Mejor alpha Lasso:", grid_lasso.best_params_)


**Nota:** si el mejor `alpha` encontrado cae justo en el borde del rango explorado (el más pequeño o el más grande de la lista), amplía `param_grid_lasso` en esa dirección y vuelve a correr — significa que el óptimo real podría estar fuera del rango probado.

In [ ]:
best_lasso = grid_lasso.best_estimator_

y_pred_lasso_train = best_lasso.predict(X_train)
y_pred_lasso_test = best_lasso.predict(X_test)

rmse_lasso_train = np.sqrt(mean_squared_error(y_train, y_pred_lasso_train))
mae_lasso_train = mean_absolute_error(y_train, y_pred_lasso_train)
r2_lasso_train = r2_score(y_train, y_pred_lasso_train)

rmse_lasso_test = np.sqrt(mean_squared_error(y_test, y_pred_lasso_test))
mae_lasso_test = mean_absolute_error(y_test, y_pred_lasso_test)
r2_lasso_test = r2_score(y_test, y_pred_lasso_test)

print(f"Train -> RMSE: {rmse_lasso_train:.4f}, MAE: {mae_lasso_train:.4f}, R²: {r2_lasso_train:.4f}")
print(f"Test  -> RMSE: {rmse_lasso_test:.4f}, MAE: {mae_lasso_test:.4f}, R²: {r2_lasso_test:.4f}")


#### Revisión de coeficientes — Lasso (variables llevadas a cero)

In [ ]:
feature_names_lasso = best_lasso.named_steps["preprocesamiento"].get_feature_names_out()
coefs_lasso = best_lasso.named_steps["modelo"].coef_

coef_lasso_df = pd.DataFrame({
    "Variable": feature_names_lasso,
    "coef_lasso": coefs_lasso
})
coef_lasso_df["coef_lasso"] = coef_lasso_df["coef_lasso"].round(4)

n_ceros = (coef_lasso_df["coef_lasso"] == 0).sum()
print(f"Variables llevadas a cero por Lasso: {n_ceros} de {len(coef_lasso_df)}")

coef_lasso_df_sorted = coef_lasso_df.sort_values("coef_lasso", key=abs, ascending=False)
coef_lasso_df_sorted.head(15)


In [ ]:
print("Variables eliminadas (coeficiente = 0):")
coef_lasso_df.loc[coef_lasso_df["coef_lasso"] == 0, "Variable"].tolist()


### Comparación: sin regularización vs. Ridge vs. Lasso

In [ ]:
comparacion_regularizacion = pd.DataFrame([
    {"Modelo": "Sin regularización", "RMSE_train": rmse_lineal_train, "MAE_train": mae_lineal_train, "R2_train": r2_lineal_train,
     "RMSE_test": rmse_lineal_test, "MAE_test": mae_lineal_test, "R2_test": r2_lineal_test},
    {"Modelo": "Ridge", "RMSE_train": rmse_ridge_train, "MAE_train": mae_ridge_train, "R2_train": r2_ridge_train,
     "RMSE_test": rmse_ridge_test, "MAE_test": mae_ridge_test, "R2_test": r2_ridge_test},
    {"Modelo": "Lasso", "RMSE_train": rmse_lasso_train, "MAE_train": mae_lasso_train, "R2_train": r2_lasso_train,
     "RMSE_test": rmse_lasso_test, "MAE_test": mae_lasso_test, "R2_test": r2_lasso_test},
])
comparacion_regularizacion


_(Completa aquí el análisis: efecto de la penalización sobre la magnitud/estabilidad de los coeficientes, y reflexión sobre las variables que Lasso eliminó — implicaciones en selección automática de características e interpretabilidad.)_

## 8. Actividad 4 — Modelo de regresión polinomial regularizado

Se combina la generación de características polinomiales con un modelo regularizado. Se utiliza Ridge por su mayor estabilidad numérica al trabajar con el número elevado de variables que genera `PolynomialFeatures`. La búsqueda de hiperparámetros explora el grado del polinomio y el parámetro de penalización.

Nota sobre el alcance de la búsqueda: se limita el grado a [1, 2] (sin incluir 3) para mantener el tiempo de ejecución razonable, dado el número de variables numéricas del conjunto de datos y el tiempo disponible para la entrega.

In [ ]:
numeric_transformer_poly_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2)),
])

preprocessor_poly_reg = ColumnTransformer(transformers=[
    ("num", numeric_transformer_poly_reg, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

pipeline_poly_ridge = Pipeline(steps=[
    ("preprocesamiento", preprocessor_poly_reg),
    ("modelo", Ridge()),
])

param_grid_poly_ridge = {
    "preprocesamiento__num__polynomial__degree": [1, 2],
    "modelo__alpha": [0.1, 1, 10, 50, 100],
}

grid_poly_ridge = GridSearchCV(
    estimator=pipeline_poly_ridge, param_grid=param_grid_poly_ridge,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)


In [ ]:
%%time
grid_poly_ridge.fit(X_train, y_train)
print("Mejor configuración:", grid_poly_ridge.best_params_)


In [ ]:
best_poly_ridge = grid_poly_ridge.best_estimator_

y_pred_poly_ridge_train = best_poly_ridge.predict(X_train)
y_pred_poly_ridge_test = best_poly_ridge.predict(X_test)

rmse_poly_ridge_train = np.sqrt(mean_squared_error(y_train, y_pred_poly_ridge_train))
mae_poly_ridge_train = mean_absolute_error(y_train, y_pred_poly_ridge_train)
r2_poly_ridge_train = r2_score(y_train, y_pred_poly_ridge_train)

rmse_poly_ridge_test = np.sqrt(mean_squared_error(y_test, y_pred_poly_ridge_test))
mae_poly_ridge_test = mean_absolute_error(y_test, y_pred_poly_ridge_test)
r2_poly_ridge_test = r2_score(y_test, y_pred_poly_ridge_test)

print(f"Train -> RMSE: {rmse_poly_ridge_train:.4f}, MAE: {mae_poly_ridge_train:.4f}, R²: {r2_poly_ridge_train:.4f}")
print(f"Test  -> RMSE: {rmse_poly_ridge_test:.4f}, MAE: {mae_poly_ridge_test:.4f}, R²: {r2_poly_ridge_test:.4f}")


_(Completa aquí: ¿la regularización permite controlar el sobreajuste al incrementar la complejidad polinomial? Compara contra el modelo polinomial sin regularizar de la Actividad 1.)_

## 9. Actividad 5 — Comparación y selección del mejor modelo

Se comparan los tres modelos construidos en las Actividades 1, 3 y 4, considerando no solo el desempeño promedio sino también su estabilidad (desviación estándar) mediante validación cruzada sobre el conjunto de entrenamiento.

In [ ]:
modelos_finales = {
    "Polinomial (Act. 1)": mejor_modelo_polinomial,
    "Ridge (Act. 3)": best_ridge,
    "Lasso (Act. 3)": best_lasso,
    "Polinomial + Ridge (Act. 4)": best_poly_ridge,
}

resumen_filas = []
for nombre, modelo in modelos_finales.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
    rmse_cv = -scores
    y_pred_test = modelo.predict(X_test)
    resumen_filas.append({
        "Modelo": nombre,
        "RMSE_CV_promedio": rmse_cv.mean(),
        "RMSE_CV_std": rmse_cv.std(),
        "RMSE_test": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "MAE_test": mean_absolute_error(y_test, y_pred_test),
        "R2_test": r2_score(y_test, y_pred_test),
    })

tabla_comparativa = pd.DataFrame(resumen_filas).sort_values("RMSE_CV_promedio")
tabla_comparativa


_(Completa aquí: selecciona el modelo final considerando desempeño (RMSE promedio), estabilidad (desviación estándar) y complejidad. Justifica la elección — por ejemplo, si dos modelos tienen desempeño similar pero uno es más estable o más simple, argumenta por qué ese es preferible. Asigna el pipeline elegido a la variable `modelo_final` en la celda de abajo, para usarlo en la Actividad 6.)_

In [ ]:
# Reemplaza el valor de la derecha por el modelo que selecciones como el mejor
# (por ejemplo: best_ridge, best_lasso, mejor_modelo_polinomial o best_poly_ridge)
modelo_final = best_ridge


## 10. Actividad 6 — Intervalos de confianza (bootstrapping)

Se aplica bootstrapping sobre el conjunto de prueba para estimar la variabilidad del RMSE y construir un intervalo de confianza del 95%. Se realizan 1000 iteraciones; en cada una se genera una muestra con reemplazo a partir de los datos de prueba, se calculan las predicciones con el modelo ya entrenado y se obtiene el RMSE correspondiente.

In [ ]:
n_iterations = 1000
stats = []

for i in range(n_iterations):
    X_resample, y_resample = resample(X_test, y_test, replace=True, n_samples=len(X_test))
    pred = modelo_final.predict(X_resample)
    rmse = np.sqrt(mean_squared_error(y_resample, pred))
    stats.append(rmse)

alpha = 0.95
p_lower = ((1.0 - alpha) / 2.0) * 100
p_upper = (alpha + (1.0 - alpha) / 2.0) * 100

lower = np.percentile(stats, p_lower)
upper = np.percentile(stats, p_upper)

print(f"Media del RMSE: {np.mean(stats):.4f}")
print(f"Desviación estándar del RMSE: {np.std(stats):.4f}")
print(f"Intervalo de confianza del RMSE (95%): [{lower:.4f}, {upper:.4f}]")

plt.figure(figsize=(8,5))
plt.hist(stats, bins=50)
plt.axvline(lower, color='red', linewidth=2)
plt.axvline(np.mean(stats), color='green', linewidth=2)
plt.axvline(upper, color='red', linewidth=2)
plt.xlabel("RMSE")
plt.ylabel("Frecuencia")
plt.title("Distribución bootstrap del RMSE - Modelo final")
plt.show()


_(Completa aquí: ¿el intervalo sugiere estabilidad o alta variabilidad? ¿Qué implicaciones tiene para la confiabilidad de las predicciones del modelo en producción?)_

## 11. Análisis de resultados

### Análisis cuantitativo

- ¿Cuál modelo obtuvo el mejor desempeño en el conjunto de test?

- ¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada? Si no coincide, ¿cuál puede ser la explicación?

- ¿El modelo con mejor métrica promedio es necesariamente el más adecuado? Justifica considerando también la desviación estándar del desempeño.

- Con base en las curvas de validación, ¿cómo cambia el error a medida que aumenta la complejidad? ¿En qué punto se evidencia sobreajuste?

- ¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?

- ¿Los intervalos de confianza obtenidos mediante bootstrapping sugieren estabilidad o alta variabilidad en el desempeño? ¿Qué implicaciones tiene esto?

### Análisis cualitativo

- ¿Qué variables fueron seleccionadas como más relevantes por el modelo Lasso?

- ¿Qué interpretación práctica tienen los coeficientes del modelo final en el contexto de la estimación de temperatura máxima?

- ¿Existen diferencias relevantes entre el modelo más preciso y el más interpretable?

- ¿Qué decisiones estratégicas podría tomar AlpesPlanck a partir de los resultados obtenidos?

- ¿Mayor precisión implica necesariamente mayor valor para la organización?

- ¿Un modelo más complejo necesariamente genera mayor valor empresarial? Discute considerando interpretabilidad, estabilidad y costo de implementación.

### Reflexión conceptual

- ¿Qué relación observas entre complejidad del modelo, capacidad de generalización y estabilidad del desempeño?

- ¿Qué fuentes de sesgo podrían estar presentes en los datos o en el proceso de modelado?

- Si el tamaño de muestra fuera mayor, ¿esperarías cambios en la estabilidad de los modelos? Explique la respuesta.

_(Responde cada pregunta con base en los resultados numéricos obtenidos arriba, una vez corras todo el notebook.)_

## 12. Uso de herramientas de IA generativa

### Declaración del uso

_(Nombre de la herramienta y tipo de uso: ayuda conceptual, generación inicial de código, depuración, redacción, etc.)_

### Prompts utilizados

_(Documenta los prompts principales que influyeron directamente en el resultado entregado.)_

### Análisis crítico del resultado

_(Responde al menos dos: ¿Qué partes del contenido generado fueron correctas y útiles? ¿Qué errores o limitaciones se identificaron? ¿Qué decisiones técnicas fueron modificadas respecto a la respuesta de la IA y por qué? ¿Qué conceptos del curso permitieron evaluar o mejorar la respuesta generada?)_

### Aportes propios del estudiante

_(¿Qué fue desarrollado, modificado o decidido por el estudiante? ¿Qué ajustes se realizaron sobre el código o la explicación original? ¿Qué aprendizajes se obtuvieron del proceso?)_